## Design a Zomato Food Delivery App

#### Functional Requirements:
1. User can search for restaurant based on location
2. User can add items to the cart.
3. User can checkout by making payments.
4. User should be notified once order placed successfully.

#### Non Functional Requirements:
1. Each part of design should be scalable and modifiable.

### Getting the key components

#### Restaurant 

1. MenuItem has code, name, cost
2. Restaurant object will have id, name, address and MenuItems list
3. Restaurant manager has list of restaurants, which can be createRes(), deleteRes(), updateRes(), searchByLocation().
4. User will have Userid, name, address, cart
5. Cart will have Restaureant, MenuItemsList, totalcost()
6. PaymentStrategy will have pay()
7. Order will have id, user, restaurant, menuItem, paymentStrategy, getOrderType()
8. Order will have subclasses which are DeliveryOrder and Pickup Order.
9. IOrderFactory for creating ScheduleOrderFactory. and NowOrderFactory.
10. Order manager will have  List of orders and can addOrder(), listOrder()
11. NotificationService will have order and notifyUser()
12. Zomato class will be orchestrator class which will communicate with all the other classes. 

### Restaurant

In [ ]:
class Restaurant:
    _next_restaurant_id = 0
    def __init__(self, id, name, location, menuItems=[]):
        self.name = name
        self.location = location
        Restaurant._next_restaurant_id += 1
        self.restaurantId =  Restaurant._next_restaurant_id
        self.menuItems: list["MenuItem"] = []

    def add_menu_item(self, item):
        self.menuItems.append(item) 

    def get_menu(self):
        return self.menuItems

### MenuItem

In [7]:
class MenuItem:
    def __init__(self, code, name, price):
        self.code = code
        self.name = name
        self.price = price
    # Getters and Setters
    def getCode(self):
        return self.code
    def setCode(self, code):
        self.code = code 
    def getPrice(self):
        return self.price
    def setPrice(self, price):
        self.price = price 

### Restaurant Manager

In [ ]:
import threading
class RestaurantManager:
    _restaurants: list["Restaurant"] = []
    _instance = None
    _lock = threading.Lock()
    def __new__(cls):
        if not cls._instance:
            with cls._lock:
                if not cls._instance:
                    cls._instance = super(RestaurantManager, cls).__new__(cls)
        return cls._instance
    def add_restaurant(cls, res):
        cls._restaurants.append(res)
    def get_restaurants(cls):
        return cls._restaurants
    def search_by_location(cls, loc):
        result: list["Restaurant"] = []
        for res in _restaurants:
            if res == loc:
                result.append(res)
        return result

res_man = RestaurantManager()
res_man.add_restaurant("kiran")
print(res_man.get_restaurants())


['kiran']


### Cart

In [10]:
class Cart:
    def __init__(self, restaurant = None, items:list[MenuItem]=[]):
        self.restaurant = restaurant
        self.items = items
    
    def add_item(self, item:MenuItem):
        self.items.append(item)
    def getTotalCost():
        total_cost = 0
        for item in items:
            total_cost += item.getPrice()
        return total_cost


### User

In [ ]:
class User:
    def __init__(self, user_id, name, address):
        self.user_id = user_id
        self.name = name
        self.address = address
        self.cart = Cart()
    # Getters and Setters

### Order

In [18]:
from abc import ABC, abstractmethod

class Order(ABC):
    _next_order_id = 0
    def __init__(self, user=None, restaurant=None, payment_strategy=None, total=0.0, scheduled=""):
        self.user = user
        self.restaurant=restaurant
        self.payment_strategy = payment_strategy
        self.scheduled = scheduled
        self.total = total
        Order._next_order_id += 1
        self.order_id = Order._next_order_id
    def process_payment(self):
        if payment_strategy:
            payment_strategy.pay(total)
            return True
        else:
            return False
    @property
    @abstractmethod
    def getType(self):
        pass

class DeliveryOrder(Order):
    def __init__(self, user=None, restaurant=None, payment_strategy=None, total=0.0, scheduled="", user_address=""):
        super().__init__(user, restaurant, payment_strategy, total, scheduled)
        self.user_address = user_address
    @property
    def getType(self):
        return "Delivery"
    def set_user_address(self, address):
        self.user_address = address
    def get_user_address(self):
        return self.user_address

class PickUpOrder(Order):
    def __init__(self, user=None, restaurant=None, payment_strategy=None, total=0.0, scheduled="", rest_address=""):
        super().__init__(user, restaurant, payment_strategy, total, scheduled)
        self.rest_address = rest_address
    @property
    def getType(self):
        return "PickUp"
    def set_rest_address(self, address):
        self.rest_address = address
    def get_rest_address(self):
        return self.rest_address


delivery_order = DeliveryOrder()
delivery_order.getType

pick_up_order = PickUpOrder()
pick_up_order.getType


'PickUp'

### Order Manager

In [23]:
import threading

class OrderManager:
    _instance = None
    _orders: list[Order] = []
    _lock = threading.Lock()
    def __new__(cls):
        if not cls._instance:
            with cls._lock:
                if not cls._instance:
                    cls._instance = super(OrderManager, cls).__new__(cls)
        return cls._instance
    def add_order(cls, order):
        cls._orders.append(order)
    def get_orders(cls):
        return cls._orders
order_manager = OrderManager()
order_manager.add_order("chicken Biryani")
order_manager.get_orders()

['chicken Biryani']

### Order factory

In [27]:
class OrderFactory(ABC):
    @abstractmethod
    def create_order(self, user, cart, restaurant, menuItems, payment_strategy, order_type):
        pass

In [ ]:
class NowOrderFactory(OrderFactory):
    def create_order(self, user, cart, restaurant, menuItems, payment_strategy, order_type):
        order = None
        if order_type == "Delivery":
            delivery_order =  DeliveryOrder(user=user, restaurant=restaurant, payment_strategy=payment_strategy)
            delivery_order.user_address = user.get_user_address()
            order = delivery_order
        else:
            pick_up_order =  PickUpOrder(user=user, restaurant=restaurant, payment_strategy=payment_strategy)
            pick_up_order.user_address = user.get_user_address()
            order = pick_up_order

In [29]:
class ScheduledOrderFactory(OrderFactory):
    def create_order(self, user, cart, restaurant, menuItems, payment_strategy, order_type):
        order = None
        if order_type == "Delivery":
            delivery_order =  DeliveryOrder(user=user, restaurant=restaurant, payment_strategy=payment_strategy)
            delivery_order.user_address = user.get_user_address()
            order = delivery_order
        else:
            pick_up_order =  PickUpOrder(user=user, restaurant=restaurant, payment_strategy=payment_strategy)
            pick_up_order.user_address = user.get_user_address()
            order = pick_up_order

### Tomato App

In [ ]:
class TomatoApp:
    def __init__(self):
        self.initialize_restaurants()

    def initialize_restaurants(self):
        restaurant1 = Restaurant("Bikaner", "Delhi")
        restaurant1.add_menu_item(MenuItem("P1", "Chole Bhature", 120))
        restaurant1.add_menu_item(MenuItem("P2", "Samosa", 15))

        restaurant2 = Restaurant("Haldiram", "Kolkata")
        restaurant2.add_menu_item(MenuItem("P1", "Raj Kachori", 80))
        restaurant2.add_menu_item(MenuItem("P2", "Pav Bhaji", 100))
        restaurant2.add_menu_item(MenuItem("P3", "Dhokla", 50))

        restaurant3 = Restaurant("Saravana Bhavan", "Chennai")
        restaurant3.add_menu_item(MenuItem("P1", "Masala Dosa", 90))
        restaurant3.add_menu_item(MenuItem("P2", "Idli Vada", 60))
        restaurant3.add_menu_item(MenuItem("P3", "Filter Coffee", 30))

        manager = RestaurantManager()
        manager.add_restaurant(restaurant1)
        manager.add_restaurant(restaurant2)
        manager.add_restaurant(restaurant3)

    def search_restaurants(self, location: str) -> list["Restaurant"]:
        return RestaurantManager().search_by_location(location)

    def select_restaurant(self, user: "User", restaurant: "Restaurant") -> None:
        user.cart.restaurant = restaurant

    def add_to_cart(self, user: "User", item_code: str) -> None:
        restaurant = user.cart.restaurant
        if restaurant is None:
            print("Please select a restaurant first.")
            return
        for item in restaurant.get_menu():
            if item.code == item_code:
                user.cart.add_item(item)
                break

    def checkout_now(self, user: "User", order_type: str, payment_strategy) -> "Order":
        return self.checkout(user, order_type, payment_strategy, NowOrderFactory())

    def checkout_scheduled(self, user: "User", order_type: str, payment_strategy, schedule_time: str) -> "Order":
        return self.checkout(user, order_type, payment_strategy, ScheduledOrderFactory(schedule_time))

    def checkout(self, user: "User", order_type: str, payment_strategy, order_factory) -> "Order | None":
        if user.cart.is_empty():
            return None

        cart = user.cart
        order = order_factory.create_order(
            user, cart, cart.restaurant, cart.items, payment_strategy, cart.get_total_cost(), order_type
        )
        OrderManager().add_order(order)
        return order

    def pay_for_order(self, user: "User", order: "Order") -> None:
        if order.process_payment():
            NotificationService.notify(order)
            user.cart.clear()

    def print_user_cart(self, user: "User") -> None:
        print("Items in cart:")
        print("------------------------------------")
        for item in user.cart.items:
            print(f"{item.code} : {item.name} : ₹{item.price}")
        print("------------------------------------")
        print(f"Grand total : ₹{user.cart.get_total_cost()}")


In [ ]:
def main():
    # Create TomatoApp object
    tomato = TomatoApp()

    # Simulate a user coming in (Happy Flow)
    user = User(101, "Aditya", "Delhi")
    print(f"User: {user.name} is active.")

    # User searches for restaurants by location
    restaurant_list = tomato.search_restaurants("Delhi")

    if not restaurant_list:
        print("No restaurants found!")
        return

    print("Found Restaurants:")
    for restaurant in restaurant_list:
        print(f" - {restaurant.name}")

    # User selects a restaurant
    tomato.select_restaurant(user, restaurant_list[0])
    print(f"Selected restaurant: {restaurant_list[0].name}")

    # User adds items to the cart
    tomato.add_to_cart(user, "P1")
    tomato.add_to_cart(user, "P2")

    tomato.print_user_cart(user)

    # User checks out the cart
    order = tomato.checkout_now(user, "Delivery", UpiPaymentStrategy("1234567890"))

    # User pays; if successful, notification is sent
    tomato.pay_for_order(user, order)


if __name__ == "__main__":
    main()
